In [ ]:
# Step 1: Multimodal Fusion PyTorch System
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class MultimodalFusionSystem(nn.Module):
    def __init__(self, embed_dim=256, num_classes=3):
        super().__init__()
        self.proj_text = nn.Linear(768, embed_dim)
        self.proj_audio = nn.Linear(16, embed_dim)
        self.proj_image = nn.Linear(768, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.fc1 = nn.Linear(embed_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, t, a, v):
        proj_t = self.proj_text(t)
        proj_a = self.proj_audio(a)
        proj_v = self.proj_image(v)
        stacked = torch.stack([proj_t, proj_a, proj_v], dim=1)
        attn_out, _ = self.attention(stacked, stacked, stacked)
        fused = torch.mean(attn_out, dim=1)
        x = self.dropout(self.relu(self.fc1(fused)))
        return self.classifier(x)

In [ ]:
# Step 2: Multimodal Dataset Simulation
class SyntheticMultimodalDataset(Dataset):
    def __init__(self, num_samples=2000):
        np.random.seed(42)
        labels = np.random.choice([0, 1, 2], size=num_samples, p=[0.35, 0.30, 0.35])
        text_feats, audio_feats, vision_feats = [], [], []
        
        for lbl in labels:
            if lbl == 2:
                t = np.random.randn(768) * 0.2 + 0.9
                a = np.random.randn(16) * 0.2 + 0.8
                v = np.random.randn(768) * 0.2 + 0.9
            elif lbl == 0:
                t = np.random.randn(768) * 0.2 - 0.9
                a = np.random.randn(16) * 0.2 - 0.8
                v = np.random.randn(768) * 0.2 - 0.9
            else:
                t = np.random.randn(768) * 0.1
                a = np.random.randn(16) * 0.1
                v = np.random.randn(768) * 0.1
            text_feats.append(t)
            audio_feats.append(a)
            vision_feats.append(v)
            
        self.t = torch.tensor(np.array(text_feats), dtype=torch.float32)
        self.a = torch.tensor(np.array(audio_feats), dtype=torch.float32)
        self.v = torch.tensor(np.array(vision_feats), dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return {'text': self.t[idx], 'audio': self.a[idx], 'vision': self.v[idx], 'label': self.labels[idx]}

dataset = SyntheticMultimodalDataset(num_samples=2000)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
# Step 3: Multimodal Training & Metric Tracking
model = MultimodalFusionSystem().to(device)
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

epochs = 8
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, epochs + 1):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        t, a, v, y = batch['text'].to(device), batch['audio'].to(device), batch['vision'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(t, a, v)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * y.size(0)
        correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total += y.size(0)
        
    tr_loss = t_loss / total
    tr_acc = (correct / total) * 100
    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            t, a, v, y = batch['text'].to(device), batch['audio'].to(device), batch['vision'].to(device), batch['label'].to(device)
            logits = model(t, a, v)
            loss = criterion(logits, y)
            v_loss += loss.item() * y.size(0)
            v_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            v_total += y.size(0)
            
    va_loss = v_loss / v_total
    va_acc = (v_correct / v_total) * 100
    val_losses.append(va_loss)
    val_accs.append(va_acc)
    
    print(f"Epoch [{epoch}/{epochs}] - Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}% | Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.2f}%")

In [ ]:
# Step 4: Plot Multimodal Training Graphs & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), train_losses, label='Train Loss', marker='o', color='#ef4444')
axes[0].plot(range(1, epochs + 1), val_losses, label='Validation Loss', marker='s', color='#f59e0b')
axes[0].set_title('Multimodal Fusion: Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

axes[1].plot(range(1, epochs + 1), train_accs, label='Train Accuracy', marker='o', color='#3b82f6')
axes[1].plot(range(1, epochs + 1), val_accs, label='Validation Accuracy', marker='s', color='#10b981')
axes[1].set_title('Multimodal Fusion: Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

# Confusion Matrix
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        t, a, v, y = batch['text'].to(device), batch['audio'].to(device), batch['vision'].to(device), batch['label'].to(device)
        logits = model(t, a, v)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Multimodal Fusion Test Set Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("MULTIMODAL FUSION CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Neutral', 'Positive']))